> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 2 · Notebook 01 — Linear algebra for portfolios

**Sessions:** S1 (Linear algebra for portfolios) · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Compute portfolio volatility from a covariance matrix.
2. Find the hidden factors in returns with PCA.
3. Simulate correlated returns with a Cholesky factor.
4. See why some covariance matrices are dangerous to invert.

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

## 1. Covariance matrix and portfolio volatility

$\sigma_p = \sqrt{w^\top \Sigma w \cdot 252}$ for daily returns.

In [ ]:
cov = rets.cov()                   # daily covariance matrix (10 × 10)
cov.round(6)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
w = np.full(len(cov), 1 / len(cov))          # equal weights
port_vol = np.sqrt(w @ cov.to_numpy() @ w * 252)
port_vol = p.check("portfolio volatility", port_vol, p.portfolio_vol(w, cov))

In [ ]:
single = rets.std() * np.sqrt(252)
print(f"Average single-asset volatility: {single.mean():.1%}")
print(f"Equal-weight portfolio volatility: {port_vol:.1%}   (diversification benefit)")
print(f"Check from the portfolio's own return series: {(rets @ w).std() * np.sqrt(252):.1%}")

## 2. PCA: hidden factors in returns

Eigen-decompose the correlation matrix. The largest eigenvalue's share of the total is the variance explained by the first principal component (usually "the market").

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
corr = rets.corr().to_numpy()
eigval, eigvec = np.linalg.eigh(corr)
eigval, eigvec = eigval[::-1], eigvec[:, ::-1]
explained = eigval[:3] / eigval.sum()
explained = p.check("PCA: variance explained by top 3", explained, p.pca_explained(rets)[0])

In [ ]:
_, loadings = p.pca_explained(rets, k=3)
pc1 = pd.Series(loadings[:, 0], index=rets.columns)
pc1 = pc1 * np.sign(pc1.sum())                     # an eigenvector's sign is arbitrary: make PC1 mostly positive
ax = pc1.sort_values().plot.barh(title="PC1 loadings (the 'market' factor)")
ax.axvline(0, color="#52514e", lw=1); plt.show()
calm, crisis = rets.loc["2019"], rets.loc["2020-02-15":"2020-06-30"]
print(f"PC1 explains {p.pca_explained(calm)[0][0]:.0%} in 2019 vs {p.pca_explained(crisis)[0][0]:.0%} in the 2020 crisis window")

## 3. Simulating correlated returns (Cholesky)

If $L L^\top = \Sigma$ and $z \sim N(0, I)$, then $L z \sim N(0, \Sigma)$.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
L = np.linalg.cholesky(cov.to_numpy())
L = p.check("Cholesky factor", L, np.linalg.cholesky(cov.to_numpy()))

In [ ]:
z = np.random.default_rng(0).standard_normal((200_000, len(cov)))
sim = z @ L.T
err = np.abs(np.cov(sim.T) - cov.to_numpy()).max() / np.abs(cov.to_numpy()).max()
print(f"Largest covariance error of the simulation, relative to the largest covariance: {err:.2%}")

## 4. Ill-conditioned covariance matrices

The condition number (largest / smallest eigenvalue) tells you how unstable $\Sigma^{-1}$ is. It grows as you add assets relative to the amount of data.

In [ ]:
window = rets.loc["2023":"2024"]
for n in [3, 5, 10]:
    c = np.linalg.cond(window.iloc[:, :n].cov())
    print(f"{n:>2} assets, {len(window)} days: condition number {c:,.0f}")

## Questions
1. Why is the portfolio volatility lower than the average single-asset volatility? When would it not be?
2. What does PC1's share rising in a crisis tell you about diversification when you need it most?
3. Why does a high condition number lead to extreme weights in mean-variance optimization (Part 8)?